In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import accuracy_score
import pickle, joblib

In [31]:
df = pd.read_csv('/content/sample_data/TOI_2025_cleaned_dataset.csv')

In [32]:
df.head()

,toi,tid,tfopwg_disp,ra,dec,pl_rade,pl_orbper,pl_trandurh,pl_trandep,pl_insol,...,in_habitable_zone,is_planet,planet_size_category_Earth-sized,planet_size_category_Jupiter-sized,planet_size_category_Neptune-sized,planet_size_category_Super-Earth,star_temp_category_G-dwarf,star_temp_category_Hot-star,star_temp_category_K-dwarf,star_temp_category_M-dwarf
0,1000.01,50365310,FP,112.357708,-12.695960,5.818163,2.171348,2.01722,656.886099,22601.948581,...,0,0,False,False,True,False,False,True,False,False
1,1001.01,88863718,PC,122.580465,-5.513852,11.215400,1.931646,3.16600,1286.000000,44464.500000,...,0,1,False,True,False,False,False,True,False,False
2,1002.01,124709665,FP,104.726966,-10.580455,23.752900,1.867557,1.40800,1500.000000,2860.610000,...,0,0,False,True,False,False,False,True,False,False
3,1003.01,106997505,FP,110.559945,-25.207017,10.546800,2.743230,3.16700,383.410000,1177.360000,...,0,0,False,True,False,False,False,False,True,False
4,1004.01,238597883,FP,122.178195,-48.802811,11.311300,3.573014,3.37000,755.000000,54679.300000,...,0,0,False,True,False,False,False,True,False,False


In [33]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [34]:
cols_to_drop = ['is_planet', 'tfopwg_disp', 'toi', 'tid', 'toi_created', 'rowupdate']

x = df.drop(columns=cols_to_drop)
y = df['is_planet']

In [35]:
model_setup = {
    "Random Forest": {
        "model": RandomForestClassifier(n_estimators=100, random_state=42),
        "desc": "An ensemble of decision trees using bagging to reduce variance."
    },
    "XGBoost": {
        "model": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
        "desc": "Gradient boosted decision trees optimized for speed and performance."
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(max_depth=10, random_state=42),
        "desc": "A single tree structure that splits data based on feature importance."
    }
}

trained_bundle = {}
performance_data = []

for name, info in model_setup.items():
    print(f"Training {name}...")
    model = info["model"]
    model.fit(X_train, y_train)

    # Make predictions for scoring
    y_pred = model.predict(X_test)

    # Calculate the 5 specific metrics
    stats = {
        "model_name": name,
        "model_description": info["desc"],
        "model_accuracy": round(accuracy_score(y_test, y_pred), 4),
        "model_precision": round(precision_score(y_test, y_pred), 4),
        "model_recall": round(recall_score(y_test, y_pred), 4),
        "model_f1_score": round(f1_score(y_test, y_pred), 4)
    }

    performance_data.append(stats)
    trained_bundle[name] = model

# Save the models for the API
joblib.dump(trained_bundle, "multi_model_classifier.pkg")

# Convert stats to DataFrame for a quick view
summary_df = pd.DataFrame(performance_data)
print("\n--- Model Training Summary ---")
print(summary_df)

Training Random Forest...
Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:53:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training Decision Tree...

--- Model Training Summary ---
      model_name                                  model_description  \
0  Random Forest  An ensemble of decision trees using bagging to...   
1        XGBoost  Gradient boosted decision trees optimized for ...   
2  Decision Tree  A single tree structure that splits data based...   

   model_accuracy  model_precision  model_recall  model_f1_score  
0          0.8346           0.8510        0.9544          0.8998  
1          0.8301           0.8502        0.9485          0.8967  
2          0.7835           0.8369        0.8962          0.8655  
